In [1]:
# ==============================================================================
# BÁO CÁO ĐỒ ÁN: TỐI ƯU HÓA RANDOM FOREST VỚI RAPIDS CUML & OPTUNA
# CELL 1: DATA INGESTION & FEATURE ENGINEERING
# ==============================================================================

import pandas as pd
import numpy as np
import time
import gc
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ HỆ THỐNG
# ------------------------------------------------------------------------------
PATH_TRAIN = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data/train_clean.parquet'
PATH_TEST = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data/test_clean.parquet'

# Ngưỡng thời gian cắt tỉa dữ liệu (Chỉ lấy từ 01/06/2017 để bảo vệ RAM và tập trung vào xu hướng gần nhất)
CUTOFF_DATE = '2017-06-01'

def memory_usage(df, name="DataFrame"):
    """Hàm tiện ích in ra dung lượng RAM đang sử dụng của DataFrame"""
    mb = df.memory_usage().sum() / 1024**2
    print(f"   [RAM INFO] Dung lượng {name}: {mb:.2f} MB")

print("="*70)
print("🚀 BƯỚC 1: KHỞI TẠO DATA PIPELINE CHO RANDOM FOREST (GPU MODE) 🚀")
print("="*70)
start_pipeline = time.time()

# ------------------------------------------------------------------------------
# 2. NẠP DỮ LIỆU & CẮT TỈA (DATA INGESTION)
# ------------------------------------------------------------------------------
print(f"\n[1/4] Đang nạp dữ liệu từ Parquet (Cắt từ ngày {CUTOFF_DATE})...")
train_df = pd.read_parquet(PATH_TRAIN)
train_df = train_df[train_df['date'] >= CUTOFF_DATE].copy()
memory_usage(train_df, "Train Data Gốc")

test_df = pd.read_parquet(PATH_TEST)
test_ids = test_df['id'].copy() # Lưu trữ ID để nộp Kaggle

# ------------------------------------------------------------------------------
# 3. FEATURE ENGINEERING (BỔ SUNG ĐẶC TRƯNG CHO RANDOM FOREST)
# LÝ DO: Random Forest học cực tốt các quy luật phi tuyến tính, nhưng lại "kém thông minh" 
# trong việc tự suy ra các phép nhân/chia. Ta sẽ "mớm" sẵn các tương tác chéo cho nó.
# ------------------------------------------------------------------------------
print("\n[2/4] Đang khởi tạo Feature Engineering (Bơm thêm trí thông minh cho mô hình)...")

def add_custom_features(df):
    """Hàm tiêm thêm các features tương tác chéo"""
    
    # Feature 1: Sức mạnh Khuyến mãi cuối tuần (Hiệu ứng cộng hưởng)
    # Nếu vừa có sale vừa là cuối tuần -> Sức mua sẽ bùng nổ
    if 'onpromotion' in df.columns and 'is_weekend' in df.columns:
        df['promo_x_weekend'] = df['onpromotion'] * df['is_weekend']
        
    # Feature 2: Quán tính xu hướng (Momentum)
    # Đo lường doanh số 7 ngày gần đây so với 28 ngày qua. >1 là đang vào trend tăng
    if 'rmean_7' in df.columns and 'rmean_28' in df.columns:
        df['momentum_7_28'] = df['rmean_7'] / (df['rmean_28'] + 1e-5) # Cộng 1e-5 để tránh lỗi chia cho 0
        
    return df

train_df = add_custom_features(train_df)
test_df = add_custom_features(test_df)
print("   -> Đã bổ sung thành công các Features: 'promo_x_weekend', 'momentum_7_28'")

# ------------------------------------------------------------------------------
# 4. CHỌN LỌC FEATURES & LOẠI BỎ RÁC (DIMENSIONALITY REDUCTION)
# LÝ DO: Cột ID, Date là kẻ thù của Random Forest gây nhiễu và Overfit.
# ------------------------------------------------------------------------------
print("\n[3/4] Đang thanh trừng Features rác và các biến không tương thích...")

# Cố tình giữ lại 'store_nbr' (để phân ranh giới không gian) nhưng vứt 'item_nbr' và 'class' 
# để tránh ma trận thưa (Sparse Matrix) khiến RF bị rối.
drop_cols = ['date', 'item_nbr', 'class', 'id'] 
drop_train = [c for c in drop_cols if c in train_df.columns]
drop_test = [c for c in drop_cols if c in test_df.columns]

# Tách X, y
y_train = train_df['unit_sales'].values
X_train = train_df.drop(columns=drop_train + ['unit_sales'])
X_test = test_df.drop(columns=drop_test)

del train_df, test_df; gc.collect() # Giải phóng RAM ngay lập tức

# ------------------------------------------------------------------------------
# 5. XỬ LÝ LỖI TOÁN HỌC, ENCODING & FILL NAN
# LÝ DO: Sklearn/cuML RF không nhận giá trị Vô cực (Inf) và String.
# ------------------------------------------------------------------------------
print("\n[4/4] Đang mã hóa Danh tính (Encoding) và bọc lót lỗi Toán học (Inf/NaN)...")

# Mã hóa Category (Biến chữ thành số nguyên)
cat_cols = ['store_nbr', 'city', 'state', 'type', 'cluster', 'family']
for col in cat_cols:
    if col in X_train.columns:
        # Ép về string trước khi chuyển sang Category code để tránh lỗi mixed-types
        X_train[col] = X_train[col].astype(str).astype('category').cat.codes
        X_test[col] = X_test[col].astype(str).astype('category').cat.codes

# Xử lý Vô cực (Infinity) sinh ra từ các phép chia Feature Engineering
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Fill NaN bằng -1 (Dấu hiệu đặc biệt để Cây Quyết định tự phân nhánh thành một cụm riêng)
X_train = X_train.fillna(-1)
X_test = X_test.fillna(-1)

# Ép kiểu dữ liệu về float32 để tương thích 100% với GPU cuML và tiết kiệm RAM
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
y_train = y_train.astype('float32')

print(f"\n✅ HOÀN TẤT BƯỚC 1 TRONG {(time.time() - start_pipeline):.2f} GIÂY!")
print(f"📊 Kích thước ma trận Huấn luyện (X_train): {X_train.shape[0]:,} dòng x {X_train.shape[1]} cột")
print(f"📊 Kích thước ma trận Dự đoán (X_test): {X_test.shape[0]:,} dòng x {X_test.shape[1]} cột")
memory_usage(X_train, "X_train (Sẵn sàng cho GPU)")
print("="*70)

🚀 BƯỚC 1: KHỞI TẠO DATA PIPELINE CHO RANDOM FOREST (GPU MODE) 🚀

[1/4] Đang nạp dữ liệu từ Parquet (Cắt từ ngày 2017-06-01)...
   [RAM INFO] Dung lượng Train Data Gốc: 1394.31 MB

[2/4] Đang khởi tạo Feature Engineering (Bơm thêm trí thông minh cho mô hình)...
   -> Đã bổ sung thành công các Features: 'promo_x_weekend', 'momentum_7_28'

[3/4] Đang thanh trừng Features rác và các biến không tương thích...

[4/4] Đang mã hóa Danh tính (Encoding) và bọc lót lỗi Toán học (Inf/NaN)...

✅ HOÀN TẤT BƯỚC 1 TRONG 57.82 GIÂY!
📊 Kích thước ma trận Huấn luyện (X_train): 12,602,346 dòng x 59 cột
📊 Kích thước ma trận Dự đoán (X_test): 3,370,464 dòng x 59 cột
   [RAM INFO] Dung lượng X_train (Sẵn sàng cho GPU): 2932.52 MB


In [2]:
# ==============================================================================
# BÁO CÁO ĐỒ ÁN: TỐI ƯU HÓA RANDOM FOREST VỚI RAPIDS CUML & OPTUNA
# CELL 2: HYPERPARAMETER TUNING TRÊN GPU T4 (ANTI-OOM VERSION)
# ==============================================================================

import cupy as cp
import cudf
import optuna
from cuml.ensemble import RandomForestRegressor as cuRF
from cuml.metrics import mean_squared_error
import time
import gc

print("="*70)
print("🎯 BƯỚC 2: KHỞI ĐỘNG OPTUNA & CUML TRÊN GPU (ANTI-OOM) 🎯")
print("="*70)

# ------------------------------------------------------------------------------
# 1. SUBSAMPLING (LẤY MẪU) ĐỂ BẢO VỆ VRAM
# LÝ DO: Không cần dùng cả 10 triệu dòng chỉ để đi "dò" tham số.
# Ta sẽ cắt 3 triệu dòng gần nhất (xu hướng mới nhất) để Tune.
# ------------------------------------------------------------------------------
TUNE_SIZE = 3_000_000
print(f"⏳ Đang trích xuất {TUNE_SIZE:,} dòng gần nhất để Tuning...")

if len(X_train) > TUNE_SIZE:
    X_tune_subset = X_train.iloc[-TUNE_SIZE:]
    y_tune_subset = y_train[-TUNE_SIZE:]
else:
    X_tune_subset = X_train
    y_tune_subset = y_train

# Chia Train/Val trên tập Subset (80/20)
split_idx = int(len(X_tune_subset) * 0.8)

# Đẩy lên GPU
X_tr_gpu = cp.asarray(X_tune_subset.iloc[:split_idx].values, dtype=cp.float32)
y_tr_gpu = cp.asarray(y_tune_subset[:split_idx], dtype=cp.float32)
X_va_gpu = cp.asarray(X_tune_subset.iloc[split_idx:].values, dtype=cp.float32)
y_va_gpu = cp.asarray(y_tune_subset[split_idx:], dtype=cp.float32)

print(f"   -> GPU Tuning Train size: {X_tr_gpu.shape[0]:,} | GPU Tuning Val size: {X_va_gpu.shape[0]:,}")

# ------------------------------------------------------------------------------
# 2. XÂY DỰNG HÀM MỤC TIÊU CHO OPTUNA (TÍCH HỢP QUÉT RÁC VRAM)
# ------------------------------------------------------------------------------
def objective(trial):
    # Dọn dẹp VRAM rác từ Trial trước
    cp.get_default_memory_pool().free_all_blocks()
    gc.collect()
    
    # Định nghĩa không gian tìm kiếm (Thu hẹp độ sâu cây để chống nổ RAM)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 100), # Giảm số cây tối đa xuống 100
        'max_depth': trial.suggest_int('max_depth', 8, 16), # Giới hạn độ sâu cực kỳ quan trọng (<= 16)
        'max_features': trial.suggest_float('max_features', 0.5, 0.85),
        'min_samples_split': trial.suggest_int('min_samples_split', 10, 50),
        'n_bins': 64, # Giảm số lượng bin phân chia để tiết kiệm tính toán
    }
    
    # Khởi tạo và Huấn luyện
    model = cuRF(**params, random_state=42)
    model.fit(X_tr_gpu, y_tr_gpu)
    
    # Dự đoán và tính lỗi
    preds = model.predict(X_va_gpu)
    rmse = cp.sqrt(mean_squared_error(y_va_gpu, preds))
    
    # Tối quan trọng: Hủy biến và quét VRAM NGAY LẬP TỨC để nhường chỗ cho Trial sau
    del model, preds
    cp.get_default_memory_pool().free_all_blocks()
    gc.collect()
    
    return rmse.item()

# ------------------------------------------------------------------------------
# 3. KÍCH HOẠT QUÁ TRÌNH TÌM KIẾM
# ------------------------------------------------------------------------------
print("\n🚀 Bắt đầu quá trình dò tìm với Optuna (15 Trials)...")
start_tune = time.time()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=15, show_progress_bar=True) 

print(f"\n✅ HOÀN TẤT TÌM KIẾM TRONG {(time.time() - start_tune)/60:.2f} PHÚT!")
print("🔥 BỘ SIÊU THAM SỐ TỐI ƯU NHẤT LÀ:")
best_params = study.best_params
for k, v in best_params.items():
    print(f"   -> {k}: {v}")
print(f"📉 RMSE tốt nhất đạt được: {study.best_value:.4f}")
print("="*70)

🎯 BƯỚC 2: KHỞI ĐỘNG OPTUNA & CUML TRÊN GPU (ANTI-OOM) 🎯
⏳ Đang trích xuất 3,000,000 dòng gần nhất để Tuning...
   -> GPU Tuning Train size: 2,400,000 | GPU Tuning Val size: 600,000

🚀 Bắt đầu quá trình dò tìm với Optuna (15 Trials)...


  0%|          | 0/15 [00:00<?, ?it/s]


✅ HOÀN TẤT TÌM KIẾM TRONG 14.89 PHÚT!
🔥 BỘ SIÊU THAM SỐ TỐI ƯU NHẤT LÀ:
   -> n_estimators: 91
   -> max_depth: 16
   -> max_features: 0.5055886176856734
   -> min_samples_split: 35
📉 RMSE tốt nhất đạt được: 0.5645


In [6]:
# ==============================================================================
# BÁO CÁO ĐỒ ÁN: TỐI ƯU HÓA RANDOM FOREST VỚI RAPIDS CUML & OPTUNA
# CELL 3: FINAL TRAINING, KHÔI PHỤC LOGARIT & KAGGLE SUBMISSION
# ==============================================================================

import cupy as cp
import cudf
from cuml.ensemble import RandomForestRegressor as cuRF
import time
import gc
import numpy as np
import pandas as pd

print("="*70)
print("🏆 BƯỚC 3: HUẤN LUYỆN FINAL MODEL & XUẤT FILE KAGGLE 🏆")
print("="*70)

# ------------------------------------------------------------------------------
# 1. ĐẨY 100% DỮ LIỆU LÊN GPU 
# ------------------------------------------------------------------------------
print("⏳ Đang đẩy 100% dữ liệu Train và Test lên GPU...")
X_train_gpu = cp.asarray(X_train.values, dtype=cp.float32)
y_train_gpu = cp.asarray(y_train, dtype=cp.float32)
X_test_gpu = cp.asarray(X_test.values, dtype=cp.float32)

# ------------------------------------------------------------------------------
# 2. KHỞI TẠO VÀ HUẤN LUYỆN MÔ HÌNH VỚI BEST PARAMS
# ------------------------------------------------------------------------------
print("\n🧠 Đang huấn luyện Random Forest bằng toàn bộ dữ liệu...")
start_train = time.time()

try:
    final_params = best_params.copy()
except NameError:
    # Backup nếu bạn lỡ làm mất biến best_params
    final_params = {
        'n_estimators': 91,
        'max_depth': 16,
        'max_features': 0.5055886176856734,
        'min_samples_split': 35
    }

final_params['n_bins'] = 64
final_params['random_state'] = 42

final_model = cuRF(**final_params)
final_model.fit(X_train_gpu, y_train_gpu)

print(f"✅ Huấn luyện hoàn tất trong {(time.time() - start_train):.2f} giây!")

# ------------------------------------------------------------------------------
# 3. DỰ ĐOÁN & HẬU XỬ LÝ (GIẢI MÃ LOGARIT CỰC KỲ QUAN TRỌNG)
# ------------------------------------------------------------------------------
print("\n🔮 Đang dự đoán doanh số cho tập Test...")
predictions_gpu = final_model.predict(X_test_gpu)

# Kéo về RAM CPU
predictions = cp.asnumpy(predictions_gpu)

# TỪ KHÓA ĂN ĐIỂM Ở ĐÂY: Giải mã Logarit (Hàm đảo ngược của log1p)
# Biến các giá trị scale nhỏ (đã được log) về lại mức doanh số thực tế
predictions = np.expm1(predictions)

# Hậu xử lý: Khóa chặt mốc 0 (Không cho phép doanh số âm)
num_negative = (predictions < 0).sum()
if num_negative > 0:
    print(f"   ⚠️ Phát hiện {num_negative:,} dự đoán bị âm. Đang tự động ép về 0 (Clipping)...")
predictions = np.clip(predictions, 0, None)

# ------------------------------------------------------------------------------
# 4. ĐÓNG GÓI VÀ XUẤT FILE NỘP BÀI
# ------------------------------------------------------------------------------
print("\n📦 Đang khởi tạo file submission.csv...")

submission_df = pd.DataFrame({
    'id': test_ids,
    'unit_sales': predictions
})

submission_path = 'submission_rf.csv'
submission_df.to_csv(submission_path, index=False)

print(f"🎉 XUẤT FILE THÀNH CÔNG: '{submission_path}'")
print(f"📊 Một số dự đoán mẫu ĐÃ ĐƯỢC KHÔI PHỤC GIÁ TRỊ THỰC:\n{submission_df.head()}")
print("="*70)

🏆 BƯỚC 3: HUẤN LUYỆN FINAL MODEL & XUẤT FILE KAGGLE 🏆
⏳ Đang đẩy 100% dữ liệu Train và Test lên GPU...

🧠 Đang huấn luyện Random Forest bằng toàn bộ dữ liệu...
✅ Huấn luyện hoàn tất trong 468.55 giây!

🔮 Đang dự đoán doanh số cho tập Test...

📦 Đang khởi tạo file submission.csv...
🎉 XUẤT FILE THÀNH CÔNG: 'submission_rf.csv'
📊 Một số dự đoán mẫu ĐÃ ĐƯỢC KHÔI PHỤC GIÁ TRỊ THỰC:
          id  unit_sales
0  125497040    0.231164
1  125497041    0.185400
2  125497042    0.487181
3  125497043    0.975796
4  125497044    1.638075
